# Weather Prediction Project: Notebook 3. Exploratory Modeling Across Multiple Families

This notebook is the exploratory modeling stage of the project. Its purpose is to broaden the search space, document the range of models tested on the rainfall prediction task, and understand how different model families behave before any final ranking decision is made.

The objective here is not to lock the final model. Instead, the notebook keeps the experimental field visible: simple baselines, neighborhood and margin-based models, ensemble trees, boosted models, and deep temporal challengers are all tested against the same forecasting target, `RainTomorrow`. What matters at this stage is learning which candidates are promising, which ones are weaker, and which ones deserve a more disciplined ranking step later in the project.

Because this notebook is exploratory, the reported holdout or test metrics are used as comparative screening evidence rather than as a pristine single-use final report. The stronger reliability claims later in the project come from rolling time-aware validation and the locked winner-package summaries, not from pretending that the common holdout block was never inspected here.

## Step 1. Import the libraries used to explore the modeling space

The exploratory stage spans several families: classical tabular models, boosted trees, static deep learning, temporal sequence models, calibration utilities, and interpretability tools. Keeping the imports explicit is useful here because the notebook is meant to show the full breadth of the modeling search rather than hiding it behind a compact summary.

In [ ]:
import ast
import json
import random
import sys
from copy import deepcopy
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
import statsmodels.api as sm
import torch
from catboost import CatBoostClassifier
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

possible_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "models").exists() and (path / "notebooks").exists()
    ),
    Path.cwd().resolve(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.features.feature_pipeline import build_features_pipeline
from src.models.ines_modeling_core import RAW_WEATHER_COLUMNS

plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.figsize"] = (10, 6)
pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_colwidth = 180


## Step 2. Load the retained artifacts and the working modeling tables

The notebook begins from the cleaned project artifacts and the prepared data tables so that every experiment is anchored to the same forecasting target. This keeps the exploratory work connected to the saved evidence used elsewhere in the project while still allowing model-by-model experimentation to remain visible.

In [ ]:
possible_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "models").exists() and (path / "notebooks").exists()
    ),
    Path.cwd().resolve(),
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models" / "final_winner_package"

aligned_df = pd.read_csv(PROCESSED_DIR / "rain_model_dataset_aligned.csv", parse_dates=["date"])
raw_weather = pd.read_csv(DATA_DIR / "raw" / "weatherAUS.csv")
raw_weather = raw_weather[RAW_WEATHER_COLUMNS].copy()
raw_weather = raw_weather[raw_weather["RainTomorrow"].notna()].drop_duplicates()

selected_aligned_features = [
    line.strip()
    for line in (PROCESSED_DIR / "ines_selected_top25_features_aligned.txt").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

comparison = pd.read_csv(MODELS_DIR / "final_model_comparison.csv")
comparison["rank"] = pd.to_numeric(comparison["rank"], errors="coerce")
threshold_curve = pd.read_csv(MODELS_DIR / "baseline_threshold_curve.csv")
calibration_methods = pd.read_csv(MODELS_DIR / "winner_calibration_methods.csv")
shap_importance = pd.read_csv(MODELS_DIR / "final_model_shap_importance.csv")
raw_curve = pd.read_csv(MODELS_DIR / "winner_uncalibrated_curve.csv")
climate_curve = pd.read_csv(MODELS_DIR / "winner_climate_regime_isotonic_curve.csv")

with open(MODELS_DIR / "final_hybrid_refinement_summary.json", "r", encoding="utf-8") as file:
    refinement_summary = json.load(file)

with open(MODELS_DIR / "winner_calibration_summary.json", "r", encoding="utf-8") as file:
    calibration_summary = json.load(file)

with open(MODELS_DIR / "dense_nn_summary.json", "r", encoding="utf-8") as file:
    dense_summary = json.load(file)

with open(MODELS_DIR / "lstm_summary.json", "r", encoding="utf-8") as file:
    lstm_summary = json.load(file)

with open(MODELS_DIR / "gru_summary.json", "r", encoding="utf-8") as file:
    gru_summary = json.load(file)

with open(MODELS_DIR / "attention_summary.json", "r", encoding="utf-8") as file:
    attention_summary = json.load(file)

baseline_result = refinement_summary["baseline_result"]
winner_features = json.loads(baseline_result["feature_names"])
catboost_params = refinement_summary["baseline_params"]
dense_best = dense_summary["best_result"]
lstm_best = lstm_summary["best_result"]
gru_best = gru_summary["best_result"]
attention_best = attention_summary["best_result"]
cnn_row = comparison.loc[comparison["model_view"] == "cnn_14_dilated"].iloc[0]

print("Project root:", PROJECT_ROOT)
print("Aligned rows:", len(aligned_df))
print("Selected aligned features:", len(selected_aligned_features))
print("Winner features:", len(winner_features))
print("Retained comparison rows:", len(comparison))


**Interpretation.** This notebook is deliberately exploratory. Each family answers a different scientific question: are linear signals already enough, do local analogues help, does tabular nonlinearity add real value, and does explicit temporal history justify the cost of deeper architectures? The purpose is not to select a winner here, but to expose where the rainfall signal becomes materially stronger and where extra complexity fails to earn its place.

At this stage, the most important reading is comparative rather than final. Models are judged by their ability to produce usable rain probabilities under a chronological split, by the balance they reach between F1 and recall, and by whether their behavior suggests a family is worth carrying forward. Final retention is postponed on purpose.

## Step 3. Define the shared helper functions used across the exploratory experiments

These helper functions keep the experiments readable without hiding the science. They control chronological splitting, probability reporting, threshold application, and the small pieces of feature preparation needed to compare several model families under the same forecasting logic.

In [ ]:
RAIN_ZONE_MAP = {
    100: "Summer dominant",
    101: "Summer",
    102: "Uniform",
    103: "Winter",
    104: "Winter dominant",
    105: "Arid",
    "100": "Summer dominant",
    "101": "Summer",
    "102": "Uniform",
    "103": "Winter",
    "104": "Winter dominant",
    "105": "Arid",
}

WIND_TO_DEG = {
    "N": 0.0,
    "NNE": 22.5,
    "NE": 45.0,
    "ENE": 67.5,
    "E": 90.0,
    "ESE": 112.5,
    "SE": 135.0,
    "SSE": 157.5,
    "S": 180.0,
    "SSW": 202.5,
    "SW": 225.0,
    "WSW": 247.5,
    "W": 270.0,
    "WNW": 292.5,
    "NW": 315.0,
    "NNW": 337.5,
}


def to_binary_target(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(int)
    return series.fillna("No").eq("Yes").astype(int)


def chronological_train_valid_test_split(df: pd.DataFrame, test_size: float = 0.20, valid_size: float = 0.20):
    ordered = df.sort_values("date").reset_index(drop=True)
    test_start = int(len(ordered) * (1 - test_size))
    train_valid = ordered.iloc[:test_start].copy()
    test_df = ordered.iloc[test_start:].copy()
    valid_start = int(len(train_valid) * (1 - valid_size))
    train_df = train_valid.iloc[:valid_start].copy()
    valid_df = train_valid.iloc[valid_start:].copy()
    return train_df, valid_df, test_df


def build_tabular_preprocessor(X: pd.DataFrame, scale_numeric: bool = True) -> ColumnTransformer:
    categorical_cols = [col for col in X.columns if not pd.api.types.is_numeric_dtype(X[col])]
    numeric_cols = [col for col in X.columns if col not in categorical_cols]

    numeric_steps = [("imputer", SimpleImputer(strategy="median", keep_empty_features=True))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipe = Pipeline(steps=numeric_steps)
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        sparse_threshold=0.0,
    )


def transform_tabular_frames(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    test_df: pd.DataFrame,
    features: list[str],
    *,
    scale_numeric: bool = True,
):
    X_train = train_df[features].copy()
    X_valid = valid_df[features].copy()
    X_test = test_df[features].copy()
    y_train = to_binary_target(train_df["rain_tomorrow"])
    y_valid = to_binary_target(valid_df["rain_tomorrow"])
    y_test = to_binary_target(test_df["rain_tomorrow"])

    preprocessor = build_tabular_preprocessor(X_train, scale_numeric=scale_numeric)
    X_train_ready = preprocessor.fit_transform(X_train)
    X_valid_ready = preprocessor.transform(X_valid)
    X_test_ready = preprocessor.transform(X_test)
    feature_names = preprocessor.get_feature_names_out()

    X_train_ready = pd.DataFrame(X_train_ready, columns=feature_names, index=X_train.index)
    X_valid_ready = pd.DataFrame(X_valid_ready, columns=feature_names, index=X_valid.index)
    X_test_ready = pd.DataFrame(X_test_ready, columns=feature_names, index=X_test.index)
    return preprocessor, X_train_ready, X_valid_ready, X_test_ready, y_train, y_valid, y_test


def find_best_threshold(y_true: pd.Series, proba: np.ndarray, thresholds: np.ndarray | None = None):
    thresholds = np.arange(0.30, 0.71, 0.02) if thresholds is None else thresholds
    rows = []
    for threshold in thresholds:
        preds = (proba >= threshold).astype(int)
        rows.append(
            {
                "threshold": float(threshold),
                "f1": float(f1_score(y_true, preds, zero_division=0)),
            }
        )
    threshold_frame = pd.DataFrame(rows)
    best_row = threshold_frame.sort_values(["f1", "threshold"], ascending=[False, True]).iloc[0]
    return float(best_row["threshold"]), threshold_frame


def print_probability_metrics(name: str, y_true: pd.Series, proba: np.ndarray, threshold: float) -> None:
    preds = (proba >= threshold).astype(int)
    print(f"--- {name} ---")
    print(f"Selected threshold: {threshold:.3f}")
    print(classification_report(y_true, preds, zero_division=0))
    print(f"ROC-AUC: {roc_auc_score(y_true, proba):.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, preds))
    print()


def parse_hidden_layers(value):
    if isinstance(value, (list, tuple)):
        return tuple(int(item) for item in value)
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, (list, tuple)):
                return tuple(int(item) for item in parsed)
        except Exception:
            digits = [part for part in value.replace('(', '').replace(')', '').replace('[', '').replace(']', '').replace(',', ' ').replace('-', ' ').split() if part.isdigit()]
            if digits:
                return tuple(int(item) for item in digits)
    return (128, 64)


def add_train_based_bins(train_df, valid_df, test_df, source_col: str, output_col: str, q: int = 5):
    quantiles = np.unique(train_df[source_col].dropna().quantile(np.linspace(0, 1, q + 1)).to_numpy())
    if len(quantiles) < 3:
        quantiles = np.unique(train_df[source_col].dropna().quantile(np.linspace(0, 1, 4)).to_numpy())
    labels = [f"bin_{idx + 1}" for idx in range(max(len(quantiles) - 1, 1))]

    def assign_bins(frame):
        return pd.cut(
            frame[source_col],
            bins=quantiles,
            labels=labels[: max(len(quantiles) - 1, 1)],
            include_lowest=True,
            duplicates="drop",
        ).astype(str)

    train_df[output_col] = assign_bins(train_df)
    valid_df[output_col] = assign_bins(valid_df)
    test_df[output_col] = assign_bins(test_df)
    return train_df, valid_df, test_df


def build_final_hybrid_frame(raw_df: pd.DataFrame, winner_feature_list: list[str]):
    feature_df = build_features_pipeline(
        raw_df,
        profile="experimental",
        use_koppen=True,
        use_seasonal_rainfall=True,
        use_temperature_humidity=True,
    )
    feature_df["date"] = pd.to_datetime(feature_df["date"])

    rename_map = {
        "dew_point_spread_9am": "dewpoint_spread_9am",
        "dew_point_spread_3pm": "dewpoint_spread_3pm",
        "day_of_year_sin": "year_cycle_sin",
        "day_of_year_cos": "year_cycle_cos",
        "rainfall_yesterday": "rainfall_yest",
    }
    feature_df = feature_df.rename(columns={k: v for k, v in rename_map.items() if k in feature_df.columns})

    feature_df["seasonal_rainfall_label"] = feature_df["seasonal_rainfall_zone"].map(RAIN_ZONE_MAP).fillna("Arid")
    feature_df["month"] = feature_df["date"].dt.month.astype("int16")
    feature_df["day"] = feature_df["date"].dt.day.astype("int16")
    feature_df["year"] = feature_df["date"].dt.year.astype("int16")

    for base in ["rainfall", "evaporation", "sunshine", "cloud_9am", "cloud_3pm", "pressure_3pm", "humidity_3pm"]:
        feature_df[f"{base}_missing_hybrid"] = feature_df[base].isna().astype(int)

    for base_col, output_col in [
        ("max_temp", "max_temp_24h_diff"),
        ("pressure_3pm", "pressure_3pm_24h_diff"),
        ("humidity_3pm", "humidity_3pm_24h_diff"),
        ("wind_gust_speed", "wind_gust_speed_24h_diff"),
    ]:
        feature_df[output_col] = feature_df.groupby("location")[base_col].diff()

    feature_df["max_temp_yest"] = feature_df.groupby("location")["max_temp"].shift(1)
    gust_deg = feature_df["wind_gust_dir"].map(WIND_TO_DEG)
    feature_df["wind_gust_dir_x"] = np.cos(np.deg2rad(gust_deg))
    feature_df["wind_gust_dir_y"] = np.sin(np.deg2rad(gust_deg))

    rainfall_zone_dummies = pd.get_dummies(feature_df["seasonal_rainfall_label"], prefix="rainfall_zone")
    rainfall_zone_dummies = rainfall_zone_dummies.astype(int)
    feature_df = pd.concat([feature_df, rainfall_zone_dummies], axis=1)
    for zone_col in [
        "rainfall_zone_Summer",
        "rainfall_zone_Summer dominant",
        "rainfall_zone_Uniform",
        "rainfall_zone_Winter",
        "rainfall_zone_Winter dominant",
    ]:
        if zone_col not in feature_df.columns:
            feature_df[zone_col] = 0

    train_df, valid_df, test_df = chronological_train_valid_test_split(feature_df)
    for source_col, output_col in [
        ("humidity_9am", "humidity_9am_bin"),
        ("pressure_9am", "pressure_9am_bin"),
        ("temp_9am", "temp_9am_bin"),
    ]:
        train_df, valid_df, test_df = add_train_based_bins(train_df, valid_df, test_df, source_col, output_col)

    hybrid_df = pd.concat([train_df, valid_df, test_df], axis=0).sort_values("date").reset_index(drop=True)
    available_features = [feature for feature in winner_feature_list if feature in hybrid_df.columns]
    missing_features = [feature for feature in winner_feature_list if feature not in hybrid_df.columns]
    return hybrid_df, available_features, missing_features


## Step 4. Prepare the aligned daily table for the first wave of candidate models

The exploratory comparison begins on the aligned daily feature table. This is the cleanest common ground for the simpler baselines and the first tree-based challengers because it lets the notebook answer an important early question: how far can we go before the richer hybrid representation becomes necessary?

The aligned top-25 list itself was fixed upstream in notebook 2 on the pre-test training period. That keeps the final test block out of shortlist construction, but it also means the validation fold used here is not fully nested with respect to feature selection. The aligned results are therefore useful screening evidence, not the strongest final reliability estimate in the project.

In [ ]:
train_aligned, valid_aligned, test_aligned = chronological_train_valid_test_split(aligned_df)
(
    aligned_preprocessor,
    X_train_aligned_ready,
    X_valid_aligned_ready,
    X_test_aligned_ready,
    y_train_aligned,
    y_valid_aligned,
    y_test_aligned,
) = transform_tabular_frames(
    train_aligned,
    valid_aligned,
    test_aligned,
    selected_aligned_features,
    scale_numeric=True,
)

X_train_valid_aligned_ready = pd.concat([X_train_aligned_ready, X_valid_aligned_ready], axis=0)
y_train_valid_aligned = pd.concat([y_train_aligned, y_valid_aligned], axis=0)

print("Aligned train period:", train_aligned["date"].min().date(), "to", train_aligned["date"].max().date())
print("Aligned validation period:", valid_aligned["date"].min().date(), "to", valid_aligned["date"].max().date())
print("Aligned test period:", test_aligned["date"].min().date(), "to", test_aligned["date"].max().date())
print("Aligned training rows:", len(train_aligned))
print("Aligned validation rows:", len(valid_aligned))
print("Aligned test rows:", len(test_aligned))


## Baseline Model 1. Logistic Regression

In [ ]:
log_reg = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
log_reg.fit(X_train_aligned_ready, y_train_aligned)

valid_prob_lr = log_reg.predict_proba(X_valid_aligned_ready)[:, 1]
lr_threshold, lr_threshold_frame = find_best_threshold(y_valid_aligned, valid_prob_lr)

log_reg_final = clone(log_reg)
log_reg_final.fit(X_train_valid_aligned_ready, y_train_valid_aligned)
test_prob_lr = log_reg_final.predict_proba(X_test_aligned_ready)[:, 1]
print_probability_metrics("Logistic Regression Report", y_test_aligned, test_prob_lr, lr_threshold)


X_train_numeric = X_train_aligned_ready.select_dtypes(include=[np.number]).astype(float)
X_train_stat = sm.add_constant(X_train_numeric, has_constant="add")
try:
    logit_results = sm.Logit(y_train_aligned, X_train_stat).fit(disp=False)
    print(logit_results.summary())
except Exception as exc:
    print("Statsmodels logistic summary could not be estimated cleanly:", exc)


**Interpretation.** Logistic Regression reaches holdout ROC-AUC 0.8620 and holdout F1 0.6300, with recall 0.6858 at the selected threshold 0.58. That is a meaningful starting point: even a mostly linear probability surface recovers a large share of rainy days, which means the cleaned aligned table already contains substantial predictive signal.

The coefficient direction is still useful qualitatively. Sunshine points in the opposite direction to rain risk, while wind gust speed, moisture stability, 3 PM pressure, and dew-point structure all move in meteorologically sensible ways. But the statsmodels summary itself is numerically unstable because the expanded encoded design contains strong collinearity and many geography dummies, so it should not be read as a clean inferential table with trustworthy standard errors or p-values.

The limitation is equally clear. Precision stays modest at 0.5826 and the model still leaves visible headroom once nonlinear interactions become available. So Logistic Regression proves that the project can build a transparent benchmark, but it also shows why the search cannot stop at a linear decision boundary. In this notebook, those holdout numbers are being used for exploratory comparison rather than as a pristine final evaluation.

## Baseline Model 2. K-Nearest Neighbors

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=25, weights="distance", metric="minkowski")
knn_model.fit(X_train_aligned_ready, y_train_aligned)

valid_prob_knn = knn_model.predict_proba(X_valid_aligned_ready)[:, 1]
knn_threshold, knn_threshold_frame = find_best_threshold(y_valid_aligned, valid_prob_knn)

knn_final = clone(knn_model)
knn_final.fit(X_train_valid_aligned_ready, y_train_valid_aligned)
test_prob_knn = knn_final.predict_proba(X_test_aligned_ready)[:, 1]
print_probability_metrics("KNN Report", y_test_aligned, test_prob_knn, knn_threshold)


**Interpretation.** K-Nearest Neighbors lands in almost the same performance band as Logistic Regression, with ROC-AUC around 0.862 and F1 around 0.64 after thresholding. That result is informative because it shows that local analogues in feature space do contain useful rainfall information: similar historical weather situations do help identify rainy days.

However, KNN does not create a convincing step forward. Its gains over the linear baseline are marginal, while its behavior depends heavily on the geometry of the transformed space and is harder to explain operationally. That makes it a valid exploratory check, but not a strong candidate for later retention.

## Baseline Model 3. Linear SVM

In [ ]:
linear_svm = CalibratedClassifierCV(
    estimator=LinearSVC(C=1.0, class_weight="balanced", random_state=42),
    method="sigmoid",
    cv=3,
)
linear_svm.fit(X_train_aligned_ready, y_train_aligned)

valid_prob_svm = linear_svm.predict_proba(X_valid_aligned_ready)[:, 1]
svm_threshold, svm_threshold_frame = find_best_threshold(y_valid_aligned, valid_prob_svm)

linear_svm_final = clone(linear_svm)
linear_svm_final.fit(X_train_valid_aligned_ready, y_train_valid_aligned)
test_prob_svm = linear_svm_final.predict_proba(X_test_aligned_ready)[:, 1]
print_probability_metrics("Linear SVM Report", y_test_aligned, test_prob_svm, svm_threshold)


**Interpretation.** The calibrated Linear SVM behaves like a second disciplined linear reference: ROC-AUC stays around 0.862 and F1 remains near the Logistic Regression level. This confirms that the problem is not simply a matter of choosing another linear classifier. Once calibration is added so that the model outputs usable probabilities, the margin-based formulation still does not open a meaningfully better decision frontier.

This is an important negative result. It tells us the rainfall task benefits from genuine nonlinearity, not just from another linear boundary with a different optimization criterion. In practical terms, the problem is already richer than a largely linear geometry can express comfortably.

## Baseline Model 4. Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced_subsample",
    max_depth=None,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42,
)
rf_model.fit(X_train_aligned_ready, y_train_aligned)

valid_prob_rf = rf_model.predict_proba(X_valid_aligned_ready)[:, 1]
rf_threshold, rf_threshold_frame = find_best_threshold(y_valid_aligned, valid_prob_rf)

rf_final = clone(rf_model)
rf_final.fit(X_train_valid_aligned_ready, y_train_valid_aligned)
test_prob_rf = rf_final.predict_proba(X_test_aligned_ready)[:, 1]
print_probability_metrics("Random Forest Report", y_test_aligned, test_prob_rf, rf_threshold)

rf_importances = (
    pd.DataFrame(
        {
            "Feature": X_train_aligned_ready.columns,
            "Gini_Importance": rf_final.feature_importances_,
        }
    )
    .sort_values("Gini_Importance", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
display(rf_importances)



**Interpretation.** Random Forest is the first experiment that makes the nonlinearity argument concrete. Its holdout ROC-AUC rises to 0.8723 and recall reaches 0.7156, outperforming the linear family and confirming that interactions among humidity, pressure, cloud, wind, and moisture variables matter in this task. The importance ranking reinforces that reading: moisture stability, dew-point spread, humidity, pressure-humidity ratio, gust speed, and sunshine all appear near the top.

Even so, the gain is not yet decisive enough to stop the search. F1 remains only around 0.64 and the bagged-tree structure still looks weaker than what the boosted families later achieve. Random Forest therefore validates the need for nonlinear modeling, but it acts more like a bridge toward stronger boosted methods than like a retained final solution.

## Baseline Model 5. Decision Tree

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=40,
    class_weight="balanced",
    random_state=42,
)
tree_model.fit(X_train_aligned_ready, y_train_aligned)

valid_prob_tree = tree_model.predict_proba(X_valid_aligned_ready)[:, 1]
tree_threshold, tree_threshold_frame = find_best_threshold(y_valid_aligned, valid_prob_tree)

tree_final = clone(tree_model)
tree_final.fit(X_train_valid_aligned_ready, y_train_valid_aligned)
test_prob_tree = tree_final.predict_proba(X_test_aligned_ready)[:, 1]
print_probability_metrics("Decision Tree Report", y_test_aligned, test_prob_tree, tree_threshold)

tree_importances = (
    pd.DataFrame(
        {
            "Feature": X_train_aligned_ready.columns,
            "Tree_Importance": tree_final.feature_importances_,
        }
    )
    .sort_values("Tree_Importance", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
display(tree_importances)


**Interpretation.** The single Decision Tree gives the clearest demonstration that interpretability alone is not sufficient. With holdout ROC-AUC 0.8450 and F1 around 0.61, it drops below the ensemble methods and shows much less stability. The learned splits are not meaningless, since humidity at 3 PM, sunshine, wind gust speed, pressure at 3 PM, and rainfall all emerge as important decision variables.

But the price of that transparency is too high. The tree over-commits to brittle partitions, loses ranking quality, and gives away predictive stability. It remains useful as an explanatory teaching device, not as a credible retained modeling option.

## Boosted Challenger 1. XGBoost

In [ ]:
scale_weight = (y_train_aligned == 0).sum() / max((y_train_aligned == 1).sum(), 1)
tscv = TimeSeriesSplit(n_splits=5)

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        eval_metric="logloss",
        scale_pos_weight=scale_weight,
        tree_method="hist",
        random_state=42,
        n_jobs=4,
    ),
    param_distributions={
        "n_estimators": [250, 350, 450, 550, 650],
        "max_depth": [3, 4, 5, 6, 7],
        "learning_rate": [0.015, 0.03, 0.05, 0.08],
        "subsample": [0.70, 0.80, 0.90, 0.95],
        "colsample_bytree": [0.65, 0.75, 0.85, 0.95],
        "min_child_weight": [1, 2, 4, 6],
        "gamma": [0.0, 0.1, 0.2, 0.4],
        "reg_lambda": [0.5, 1.0, 1.5, 2.0, 3.0],
    },
    n_iter=10,
    scoring="f1",
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=1,
)

xgb_search.fit(X_train_aligned_ready, y_train_aligned)
print("Best XGBoost parameters:", xgb_search.best_params_)
print(f"Best cross-validated F1: {xgb_search.best_score_:.4f}")

xgb_search_results = (
    pd.DataFrame(xgb_search.cv_results_)[["params", "mean_test_score", "std_test_score"]]
    .sort_values("mean_test_score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
display(xgb_search_results)


In [ ]:
xgb_model = XGBClassifier(
    eval_metric="logloss",
    scale_pos_weight=scale_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=4,
    **xgb_search.best_params_,
)
xgb_model.fit(X_train_aligned_ready, y_train_aligned)

valid_prob_xgb = xgb_model.predict_proba(X_valid_aligned_ready)[:, 1]
xgb_threshold, xgb_threshold_frame = find_best_threshold(y_valid_aligned, valid_prob_xgb)

combined_scale_weight = (y_train_valid_aligned == 0).sum() / max((y_train_valid_aligned == 1).sum(), 1)
xgb_final = XGBClassifier(
    eval_metric="logloss",
    scale_pos_weight=combined_scale_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=4,
    **xgb_search.best_params_,
)
xgb_final.fit(X_train_valid_aligned_ready, y_train_valid_aligned)
test_prob_xgb = xgb_final.predict_proba(X_test_aligned_ready)[:, 1]
print_probability_metrics("XGBoost Report", y_test_aligned, test_prob_xgb, xgb_threshold)

xgb_importances = (
    pd.DataFrame(
        {
            "Feature": X_train_aligned_ready.columns,
            "Gain_Approximation": xgb_final.feature_importances_,
        }
    )
    .sort_values("Gain_Approximation", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
display(xgb_importances)



**Interpretation.** XGBoost is the first exploratory model that clearly changes the hierarchy of the search space. In the direct aligned-table notebook run shown here, it lands around holdout ROC-AUC 0.8825 with F1 about 0.65. When the aligned challenger is later frozen for notebook 4's disciplined ranking table, the saved comparison sits at ROC-AUC 0.8842 and F1 0.6573. Either way, the conclusion is the same: boosted trees extract substantially more from the daily feature table than Logistic Regression, Linear SVM, KNN, or a single tree.

The gain profile is also informative. Humidity at 3 PM, rain-today status, missing rainfall information, moisture-stability terms, sunshine, gust speed, and coastal-location signals all appear near the top. So XGBoost does not merely score better; it shows that rainfall prediction is driven by interacting tabular signals and regime-sensitive context. That makes boosted trees the main family worth carrying forward.

### Integrated all-feature and scaled-feature model screen

The exploratory modeling stage also includes the broader all-feature and scaled-feature experiments that tested threshold search, XGBoost feature importance, SHAP interpretation, and probability calibration. These experiments are not treated as a separate project; they are folded into the model-search evidence that feeds Notebook 4.

The same caution applies here as in the rest of notebook 3: the reported holdout rows are comparative screening evidence that informed notebook 4, not a claim that the project preserved a single untouched test block throughout the whole exploration phase.

<!-- INTEGRATED_MODELING_MERGE -->
### Supplementary model-screen evidence carried into selection

This section is the explicit merged record of the broader all-feature study. It is where the additional classical-model, boosted-tree, threshold-search, SHAP, calibration, AutoML, and clustering work enters the final notebook sequence.

| Candidate from the supplementary modeling study | Representation | Decision threshold | ROC-AUC | F1 | Precision | Recall | Selection reading |
| --- | --- | ---: | ---: | ---: | ---: | ---: | --- |
| Random Forest | all-feature weather table | default | 0.8741 | 0.65 | 0.61 | 0.69 | Useful nonlinear benchmark, but weaker than the strongest boosted challengers. |
| KNN | scaled all-feature table | default | 0.8422 | 0.57 | 0.73 | 0.47 | High precision but far too many missed rainy days. |
| Linear SVM | scaled all-feature table with calibrated probabilities | default | 0.8657 | 0.25 | 0.91 | 0.15 | Too conservative for the positive class to remain a serious final candidate. |
| Decision Tree | all-feature table | default | 0.8239 | 0.59 | 0.48 | 0.75 | Recall-heavy, but too unstable and too coarse as a final model. |
| XGBoost | all-feature weather table | default | 0.8895 | 0.66 | 0.57 | 0.78 | Strong classical challenger that proves the broader feature surface is genuinely useful. |
| Calibrated XGBoost | all-feature weather table with isotonic calibration | 0.303 | 0.8881 | 0.67 | 0.62 | 0.73 | Strong enough to shape ranking, thresholding, calibration, and SHAP interpretation, but still not strong enough to become the final winner. |
| LightGBM | all-feature weather table | default | 0.8897 | 0.67 | 0.63 | 0.72 | Another credible boosted-tree study, close to XGBoost but still not a cleaner final choice than CatBoost. |
| Calibrated LightGBM | all-feature weather table with isotonic calibration | 0.293 | 0.8904 | 0.67 | 0.64 | 0.71 | Confirms that calibration helps boosted trees, but still does not overturn the final ranking logic. |
| Nearest Centroid | selected-feature scaled table | default | - | 0.53 | 0.41 | 0.73 | Interesting compact baseline, but too weak for final retention. |

Three things from this study survive directly into the later notebooks:

1. Boosted trees are genuine contenders on richer feature surfaces.
2. Threshold tuning materially changes the rain/no-rain operating point.
3. Calibration deserves its own evaluation because probability reliability and classification F1 are not identical goals.

The recurring high-importance signals in the boosted-tree study were also consistent with the rest of the project: `humidity_3pm`, `dewpoint_spread_3pm`, `rainfall`, `sunshine`, `wind_gust_speed`, `pressure_3pm`, `cloud_3pm`, `rain_today`, and climate / location context all remained prominent.

![Calibrated XGBoost feature importance](../reports/figures/integrated_evidence/integrated_calibrated_xgboost_feature_importance.png)

![XGBoost calibration before adjustment](../reports/figures/integrated_evidence/integrated_xgboost_calibration_before.png)

![XGBoost calibration after adjustment](../reports/figures/integrated_evidence/integrated_xgboost_calibration_after.png)

![XGBoost learning curve](../reports/figures/integrated_evidence/integrated_xgboost_learning_curve.png)

![XGBoost SHAP summary](../reports/figures/integrated_evidence/integrated_xgboost_shap_summary.png)

<!-- INTEGRATED_LAZYPREDICT -->
### Automated first-pass model sweep (LazyPredict)

Before the broader modeling notebook settled into explicit reruns, it used a chronological LazyPredict sweep as a first-pass triage step on the all-feature train and test tables. This was never meant to be the final ranking by itself. Its purpose was to identify which families deserved a more disciplined second pass with controlled preprocessing, threshold tuning, calibration, and interpretation.

| LazyPredict signal from the automated sweep | Representative source output | How the merged project uses it |
| --- | --- | --- |
| Nearest Centroid surfaced with the strongest balanced accuracy. | Balanced Accuracy `0.7648`, ROC-AUC `0.8421`, aggregate default-label F1 `0.7908` | Flagged the centroid family for a targeted rerun, but the later selected-feature rain-class evaluation is the comparable result used in screening and ranking. |
| XGBoost surfaced as the strongest practical nonlinear family. | Balanced Accuracy `0.7498`, ROC-AUC `0.8854`, aggregate default-label F1 `0.8475` | Motivated the later explicit XGBoost, calibrated XGBoost, and boosted-tree ranking path. |
| SVC posted high headline accuracy, but at extreme runtime cost. | Accuracy `0.8606`, ROC-AUC `0.8699`, runtime about `1125.8` seconds | Supported keeping margin-based models visible, but not as the project's main scalable path. |
| Logistic Regression and LinearSVC remained competitive in the broad sweep. | ROC-AUC about `0.8627`, balanced accuracy about `0.7127` to `0.7178` | Reinforced the decision to retain an interpretable linear benchmark in the final project. |

The crucial caveat is comparability. LazyPredict reports package-level default-label metrics on the broad all-feature split, whereas the later notebooks compare retained models with explicit rain-class threshold tuning and study-specific preprocessing surfaces. The automated sweep therefore remains family triage evidence, not direct ranking evidence.

<!-- INTEGRATED_AUX_MODELING -->
### Additional exploratory studies retained as context, not finalists

Beyond the automated first-pass sweep above, the broader modeling notebook also tested a few extra ideas that are worth preserving because they clarify what did not improve the project enough to matter later.

| Exploratory side study | Main result | Why it is kept in the merged storyline |
| --- | --- | --- |
| TPOT AutoML pipeline | average CV score about `0.6587` | Shows that automated search found usable structure, but not a cleaner or stronger final workflow than the retained models. |
| Hierarchical clustering after PCA | Adjusted Rand about `0.009`, NMI about `0.007` | Useful diagnostic check, but the clusters did not align with the rain/no-rain target strongly enough to support a predictive path. |
| KMeans on geographic / climate features | helpful for visual location grouping, but not retained as comparable classifier evidence | Confirms that location context matters, yet clustering alone is not the project's strongest predictive answer. |

These studies are merged into the story as negative evidence of a good kind: they show that the project tested broader ideas, but only kept them when they added real forecasting value.

<!-- EXPLORATION_TO_RANKING -->
### How notebook 3 hands off to the ranking notebook

Notebook 3 is deliberately broad. Its job is to test whether different model families become competitive once the right feature surface is available. Notebook 4 then takes that search space and ranks it under a stricter decision rule.

| Exploratory evidence produced here | Canonical feature surface name used later | How notebook 4 uses it |
| --- | --- | --- |
| Logistic Regression, Random Forest, and aligned XGBoost | aligned top-25 feature table | Supports the interpretable-baseline comparison on one shared footing. |
| Random Forest, Decision Tree, XGBoost, LightGBM, and calibrated boosted-tree studies | broader all-feature weather table with climate context | Supplies the richer non-retained challenger evidence and the calibration lessons. |
| Saved dense benchmark plus broader static deep architectures | 68-feature scaled high-dimensional hybrid representation or the broader scaled high-dimensional table | Shows that static neural models were tested seriously, even if they were not retained. |
| Temporal LSTM, GRU, Attention, and CNN challengers | rolling 68-driver sequence views | Supplies the strongest sequence-family representatives. |
| CatBoost hybrid candidate | 68-feature hybrid-plus-core winner representation | Becomes the strongest exploratory tabular candidate and the eventual final-model favorite. |

That transition keeps the merged project clean. Notebook 3 broadens the search space. Notebook 4 does the ranking. Notebook 5 stops model shopping and refines only the retained winner.

## Step 5. Prepare the richer hybrid representation for the strongest boosted and deep candidates

The hybrid model is not a stacked ensemble and it is not a second classifier pasted on top of the aligned table. It is a richer forecasting representation. Starting from the core weather columns, it adds spatially informed filling within the observed station network, regime-aware context, missingness flags, lag-based weather changes, cyclical seasonal terms, wind-direction geometry, and interaction features designed to capture afternoon moisture instability more directly.

This step matters because the earlier aligned experiments already showed that the problem rewards nonlinear interaction modeling. The hybrid table is therefore a scientific response to that result: instead of only changing algorithms, the project asks whether a better representation of the same meteorological day can expose more usable rainfall signal.

In [ ]:
hybrid_df, winner_features_available, missing_winner_features = build_final_hybrid_frame(raw_weather, winner_features)
train_hybrid, valid_hybrid, test_hybrid = chronological_train_valid_test_split(hybrid_df)

X_train_hybrid = train_hybrid[winner_features_available].copy()
X_valid_hybrid = valid_hybrid[winner_features_available].copy()
X_test_hybrid = test_hybrid[winner_features_available].copy()
y_train_hybrid = to_binary_target(train_hybrid["rain_tomorrow"])
y_valid_hybrid = to_binary_target(valid_hybrid["rain_tomorrow"])
y_test_hybrid = to_binary_target(test_hybrid["rain_tomorrow"])

catboost_cat_cols = [col for col in winner_features_available if not pd.api.types.is_numeric_dtype(X_train_hybrid[col])]
for frame in [X_train_hybrid, X_valid_hybrid, X_test_hybrid]:
    for column in catboost_cat_cols:
        frame[column] = frame[column].fillna("Missing").astype(str)

print("Winner features reconstructed:", len(winner_features_available))
print("Missing winner features after reconstruction:", missing_winner_features)
print("CatBoost categorical columns:", len(catboost_cat_cols))
print("Hybrid train rows:", len(train_hybrid))
print("Hybrid validation rows:", len(valid_hybrid))
print("Hybrid test rows:", len(test_hybrid))


## Boosted Challenger 2. CatBoost

In [ ]:

def catboost_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 600, step=50),
        "depth": trial.suggest_int("depth", 6, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.08, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 8.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
    }
    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        scale_pos_weight=(y_train_hybrid == 0).sum() / max((y_train_hybrid == 1).sum(), 1),
        **params,
    )
    model.fit(X_train_hybrid, y_train_hybrid, cat_features=catboost_cat_cols)
    valid_proba = model.predict_proba(X_valid_hybrid)[:, 1]
    threshold, _ = find_best_threshold(y_valid_hybrid, valid_proba)
    preds = (valid_proba >= threshold).astype(int)
    trial.set_user_attr("threshold", float(threshold))
    trial.set_user_attr("validation_roc_auc", float(roc_auc_score(y_valid_hybrid, valid_proba)))
    return float(f1_score(y_valid_hybrid, preds, zero_division=0))


catboost_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
catboost_study.enqueue_trial(catboost_params)
catboost_study.optimize(catboost_objective, n_trials=6, show_progress_bar=False)

print("Best CatBoost parameters from the seeded compact notebook search:")
print(catboost_study.best_params)
print("Best validation F1 from the seeded compact notebook search:", catboost_study.best_value)
print("Saved project winner parameters:")
print(catboost_params)


In [ ]:
catboost_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    scale_pos_weight=(y_train_hybrid == 0).sum() / max((y_train_hybrid == 1).sum(), 1),
    **catboost_params,
)
catboost_model.fit(X_train_hybrid, y_train_hybrid, cat_features=catboost_cat_cols)

valid_prob_cat = catboost_model.predict_proba(X_valid_hybrid)[:, 1]
cat_threshold, cat_threshold_frame = find_best_threshold(y_valid_hybrid, valid_prob_cat)

X_train_valid_hybrid = pd.concat([X_train_hybrid, X_valid_hybrid], axis=0).reset_index(drop=True)
y_train_valid_hybrid = pd.concat([y_train_hybrid, y_valid_hybrid], axis=0).reset_index(drop=True)
catboost_final = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    scale_pos_weight=(y_train_valid_hybrid == 0).sum() / max((y_train_valid_hybrid == 1).sum(), 1),
    **catboost_params,
)
catboost_final.fit(X_train_valid_hybrid, y_train_valid_hybrid, cat_features=catboost_cat_cols)
test_prob_cat = catboost_final.predict_proba(X_test_hybrid)[:, 1]
print_probability_metrics("CatBoost Report", y_test_hybrid, test_prob_cat, cat_threshold)

cat_importance = (
    pd.DataFrame(
        {
            "Feature": winner_features_available,
            "CatBoost_Importance": catboost_final.get_feature_importance(),
        }
    )
    .sort_values("CatBoost_Importance", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
display(cat_importance)

print("Retained final winner:")
print(
    {
        "test_roc_auc": baseline_result["test_roc_auc"],
        "test_f1": baseline_result["test_f1"],
        "test_precision": baseline_result["test_precision"],
        "test_recall": baseline_result["test_recall"],
        "selected_threshold": baseline_result["selection_threshold"],
    }
)


**Interpretation.** CatBoost becomes the most convincing boosted candidate once the hybrid representation is available. The seeded compact notebook search reaches a best validation F1 of 0.6629, and the saved retained configuration later delivers holdout ROC-AUC 0.9016, holdout F1 0.6853, and recall 0.7510. These values matter because they do not just edge out the aligned-table models; they move the frontier of the whole notebook upward.

The reason this candidate is promising is not only raw score. CatBoost handles the heterogeneous hybrid table naturally, keeps the probability output stable, and improves the balance between discrimination, recall, and precision without requiring an excessively complicated inference story. At the exploratory stage, that is exactly the signal we need: this family has earned a place in the formal ranking notebook.

## Deep Static Challenger. Dense Neural Network

<!-- INTEGRATED_STATIC_DEEP -->
### Additional static deep architectures from the broader deep-learning study

The static deep-learning notebook did not stop at one dense network. It also tested several related architectures on the scaled high-dimensional feature table, and it explicitly set up a TabNet trial so the project checked both dense and attention-style tabular deep learning rather than stopping at standard multilayer perceptrons.

| Static deep architecture | Threshold used | ROC-AUC | PR-AUC | F1 | Precision | Recall | Reading |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| Sequential dense network | 0.42 | 0.8869 | 0.7426 | 0.67 | 0.61 | 0.74 | Best of the static deep variants, but still weaker than the final CatBoost winner. |
| Wide & Deep network | 0.59 | 0.8840 | 0.7329 | 0.65 | 0.57 | 0.76 | Competitive recall, but not a cleaner final tradeoff. |
| ResNet-style dense model | 0.50 | 0.8784 | 0.7213 | 0.63 | 0.52 | 0.80 | Very recall-heavy, but too many false alarms for the final role. |
| TabNet prototype (unscored) | configuration stage only in source notebook | - | - | - | - | - | Explicitly scoped as an attention-style tabular deep check, but no locked comparable evaluation artifact was retained. |

This matters because the merged project can now say something stronger: it was not only one dense network versus CatBoost. Several static deep architectures were tried, and even the best of them did not change the final ranking enough to displace the retained winner.

<!-- INTEGRATED_TABNET_NOTE -->
#### TabNet coverage in the merged deep-learning record

The source deep-learning notebook explicitly prepared a `TabNetClassifier` configuration with `n_d=32`, `n_a=32`, `n_steps=5`, and `gamma=1.3` as the project's attention-style tabular deep-learning check. Unlike the sequential dense, Wide & Deep, and ResNet runs, that branch did not retain a final scored evaluation artifact in the repository.

The merged notebook sequence therefore keeps TabNet as documented architecture coverage rather than as a ranked quantitative challenger. That is still useful evidence: the project did not ignore attention-style tabular deep learning, but it only promotes studies into the final ranking once they produce a reproducible scored package.

In [ ]:
(
    dense_preprocessor,
    X_train_dense_ready,
    X_valid_dense_ready,
    X_test_dense_ready,
    y_train_dense,
    y_valid_dense,
    y_test_dense,
) = transform_tabular_frames(
    train_hybrid,
    valid_hybrid,
    test_hybrid,
    winner_features_available,
    scale_numeric=True,
)

dense_model = MLPClassifier(
    hidden_layer_sizes=parse_hidden_layers(dense_best["hidden_layer_sizes"]),
    activation=str(dense_best["activation"]),
    alpha=float(dense_best["alpha"]),
    learning_rate_init=float(dense_best["learning_rate_init"]),
    batch_size=int(dense_best["batch_size"]),
    solver="adam",
    early_stopping=True,
    n_iter_no_change=10,
    max_iter=200,
    random_state=42,
)
dense_model.fit(X_train_dense_ready, y_train_dense)

valid_prob_dense = dense_model.predict_proba(X_valid_dense_ready)[:, 1]
dense_threshold, dense_threshold_frame = find_best_threshold(y_valid_dense, valid_prob_dense)
dense_test_prob = dense_model.predict_proba(X_test_dense_ready)[:, 1]
print_probability_metrics("Dense Neural Network Report", y_test_dense, dense_test_prob, dense_threshold)

print("Retained dense benchmark:")
print(
    {
        "candidate": dense_best["candidate_label"],
        "test_roc_auc": dense_best["test_roc_auc"],
        "test_f1": dense_best["test_f1"],
        "validation_threshold": dense_best["validation_threshold"],
    }
)


**Interpretation.** The dense neural network confirms that the richer feature table is genuinely valuable. Its retained benchmark reaches holdout ROC-AUC 0.8995 and holdout F1 0.6659, with recall climbing to 0.8106. That very high recall is important because it shows a static deep learner can find a broad rain-detection signal in the hybrid representation and does not collapse when compared with stronger tree-based models.

At the same time, the trade-off is visible. Precision falls to about 0.5651, so the model buys recall partly by admitting more false alarms. The dense network is therefore a real exploratory challenger, but not yet a cleaner decision rule than the strongest CatBoost workflow.

## Step 6. Load the temporal-sequence utilities used by the rolling-history challengers

The exploratory notebook then broadens the search further by testing whether sequence models can recover information from recent temporal context that static tabular models may miss. The long reusable sequence utilities are kept in one dedicated project helper so the notebook can stay focused on the model-specific training blocks rather than on repeated scaffolding code.

In [ ]:
from src.models.sequence_modeling_utils import (
    LSTMSequenceClassifier,
    GRUSequenceClassifier,
    TemporalAttentionClassifier,
    TemporalCNNClassifier,
    build_sequence_splits,
    configure_sequence_modeling,
    fit_sequence_model,
    predict_sequence_probabilities,
    prepare_sequence_partitions,
)

configure_sequence_modeling(
    build_tabular_preprocessor_fn=build_tabular_preprocessor,
    to_binary_target_fn=to_binary_target,
    find_best_threshold_fn=find_best_threshold,
)


## Temporal Sequence Challenger 1. LSTM

The temporal sections below report the retained exported benchmark packages carried forward into notebooks 4 to 6. They do not retrain fresh sequence models inside notebook 3, because the saved benchmark artifacts are the comparable validation-first evidence used in the later ranking and final summary.


In [ ]:

lstm_report = pd.DataFrame(
    [
        {
            "candidate": lstm_best["candidate_label"],
            "selection_basis": lstm_summary["selection_basis"],
            "lookback_days": int(lstm_best["lookback"]),
            "validation_threshold": float(lstm_best["validation_threshold"]),
            "test_roc_auc": float(lstm_best["test_roc_auc"]),
            "test_f1": float(lstm_best["test_f1"]),
            "test_precision": float(lstm_best["test_precision"]),
            "test_recall": float(lstm_best["test_recall"]),
        }
    ]
)

display(lstm_report)
print("Retained LSTM benchmark package metrics are the exported validation-first evidence used later in notebooks 4 to 6.")


## Temporal Sequence Challenger 2. GRU

In [ ]:

gru_report = pd.DataFrame(
    [
        {
            "candidate": gru_best["candidate_label"],
            "selection_basis": gru_summary["selection_basis"],
            "lookback_days": int(gru_best["lookback"]),
            "validation_threshold": float(gru_best["validation_threshold"]),
            "test_roc_auc": float(gru_best["test_roc_auc"]),
            "test_f1": float(gru_best["test_f1"]),
            "test_precision": float(gru_best["test_precision"]),
            "test_recall": float(gru_best["test_recall"]),
        }
    ]
)

display(gru_report)
print("Retained GRU benchmark package metrics are the exported validation-first evidence used later in notebooks 4 to 6.")


## Temporal Sequence Challenger 3. Attention

In [ ]:

attention_report = pd.DataFrame(
    [
        {
            "candidate": attention_best["candidate_label"],
            "selection_basis": attention_summary["selection_basis"],
            "lookback_days": int(attention_best["lookback"]),
            "validation_threshold": float(attention_best["validation_threshold"]),
            "test_roc_auc": float(attention_best["test_roc_auc"]),
            "test_f1": float(attention_best["test_f1"]),
            "test_precision": float(attention_best["test_precision"]),
            "test_recall": float(attention_best["test_recall"]),
        }
    ]
)

display(attention_report)
print("Retained attention benchmark package metrics are the exported validation-first evidence used later in notebooks 4 to 6.")


## Temporal Sequence Challenger 4. CNN

In [ ]:

cnn_report = pd.DataFrame(
    [
        {
            "candidate": "Temporal CNN 14-day dilated",
            "selection_basis": str(cnn_row["selection_basis"]),
            "lookback_days": 14,
            "validation_threshold": float(cnn_row["selection_threshold"]),
            "test_roc_auc": float(cnn_row["test_roc_auc"]),
            "test_f1": float(cnn_row["test_f1"]),
            "test_precision": float(cnn_row["test_precision"]),
            "test_recall": float(cnn_row["test_recall"]),
        }
    ]
)

display(cnn_report)
print("Retained CNN benchmark package metrics are the exported validation-first evidence used later in notebooks 4 to 6.")



**Interpretation.** The temporal challengers still make the exploratory conclusion much stronger. The retained exported LSTM reaches holdout F1 0.6836, the GRU 0.6826, the attention model 0.6796, and the best CNN 0.6824. In other words, once recent station history is modeled explicitly, several architectures come very close to the best tabular winner. That is evidence that temporal context is real and useful in this problem for the same station network on which the sequences were built.

But closeness is not the same as displacement. None of these sequence models produces a clean enough overall gain to make the boosted hybrid approach obsolete, and each one adds substantial training and tuning cost. The exploratory lesson is therefore nuanced: temporal structure matters, yet the best overall decision may still come from a stronger tabular representation rather than from the most complex architecture.

Using the retained benchmark packages directly also keeps notebook 3 methodologically clean. The later ranking notebooks are now reading the same exported sequence evidence that appears here, instead of comparing a fresh in-notebook rerun against a separate saved benchmark package.


The exploratory phase broadens the search space and clarifies what actually drives performance. Linear models give a credible interpretability anchor, Random Forest shows that nonlinearity matters, aligned and all-feature XGBoost confirm that boosted trees are the strongest classical family, and the hybrid CatBoost plus neural challengers show that richer representation and temporal context both create real gains.

That is enough to leave exploration behind. The next notebook does not rerun the search; it ranks the merged candidates under explicit retention criteria, keeps the feature surfaces visible, and decides which models remain active in the final workflow.

## Step 7. Integrated alternate-branch evidence

The two appendices below fold the supplementary pruning and dense-network evidence directly into Notebook 3. They are kept after the main modeling sequence so the primary workflow remains readable while the extra evidence stays in the same notebook.


### Appendix A. Integrated Alternate Branch: XGBoost Gain Pruning

This appendix integrates the newer XGBoost pruning notebook into Notebook 3. It preserves the parallel branch evidence: the selected-feature CSV workflow, feature-importance pruning, native-NaN tree modeling, and the CatBoost pruned-surface cross-check. Large modeling CSVs remain local; the executed outputs and exported figures preserve the evidence used in the report and Streamlit app.


In [ ]:
#pip install -r requirements.txt
%load_ext autoreload
%autoreload 2
#importing necessary packages
import pandas as pd
import sys
from pathlib import Path
import src.features.build_features as ft 
%matplotlib inline
import matplotlib.pyplot as plt
from src.config import TRAIN_FINAL, TEST_FINAL
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train = pd.read_csv("../data/preprocessed/selected_features_train.csv")
test = pd.read_csv("../data/preprocessed/selected_features_test.csv")

In [ ]:
# split X, y
X_train = train.drop(columns=['rain_tomorrow'])
y_train = train['rain_tomorrow']

X_test = test.drop(columns=['rain_tomorrow'])
y_test = test['rain_tomorrow']

Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
y_prob_lr = log_reg.predict_proba(X_test)[:, 1]

print("--- Logistic Regression Report ---")
print(classification_report(y_test, y_pred_lr))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}")
print()

X_train_numeric = X_train.select_dtypes(include=[np.number])
X_train_numeric = X_train_numeric.astype(float)
import statsmodels.api as sm
X_train_stat = sm.add_constant(X_train_numeric)
logit_results = sm.Logit(y_train, X_train_stat).fit()

print(logit_results.summary())

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', 
                                  max_depth=12, n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("--- Random Forest Report ---")
print(classification_report(y_test, y_pred_rf))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}")

importances_rf = pd.DataFrame({
    'Feature': X_train.columns,
    'Gini_Importance': rf_model.feature_importances_
}).sort_values(by='Gini_Importance', ascending=False)

print("--- Random Forest Gini Importance ---")
print(importances_rf.head(20))

KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=11, n_jobs=-1)
knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)
y_prob_knn = knn_model.predict_proba(X_test)[:, 1]

print("--- k-NN Report ---")
print(classification_report(y_test, y_pred_knn))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_knn):.4f}")

SVM LinearSVC

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

base_svc = LinearSVC(class_weight='balanced', max_iter=2000, random_state=42)
svc_model = CalibratedClassifierCV(base_svc)
svc_model.fit(X_train, y_train)

y_pred_svc = svc_model.predict(X_test)
y_prob_svc = svc_model.predict_proba(X_test)[:, 1]

print("--- Linear SVM Report ---")
print(classification_report(y_test, y_pred_svc))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_svc):.4f}")

Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier


dt_model = DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
y_prob_dt = dt_model.predict_proba(X_test)[:, 1]

print("--- Decision Tree Report ---")
print(classification_report(y_test, y_pred_dt))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_dt):.4f}")

# Feature Importance
dt_importances = pd.Series(dt_model.feature_importances_, index=X_train.columns)
print("\nTop 15 Features:")
print(dt_importances.nlargest(15))

XGBoost

In [ ]:
## cross validation (time series) and find best params (randomized search)

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
import xgboost as xgb

scale_weight= np.sum(y_train == 0) / np.sum(y_train == 1)

tscv = TimeSeriesSplit(n_splits=5)

model = xgb.XGBClassifier(
    use_label_encoder=False, 
    eval_metric='logloss',
    scale_pos_weight=scale_weight # important for class imbalance
)

param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0, 0.1, 0.2]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=15,           # tries 15 random combinations
    scoring='f1',        # optimizes on f1 score
    cv=tscv,             # using TimeSeriesSplit
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print(f"best params: {random_search.best_params_}")
print(f"best f1 score: {random_search.best_score_}")

results_df = pd.DataFrame(random_search.cv_results_)

fold_cols = [col for col in results_df.columns if col.startswith('split')]
important_cols = ['params', 'mean_test_score', 'std_test_score'] + fold_cols

best_run_results = results_df[results_df['rank_test_score'] == 1][important_cols]

print("--- performance per fold ---")
print(best_run_results.T)

In [ ]:
from xgboost import XGBClassifier

ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_model = XGBClassifier(n_estimators=100, scale_pos_weight=ratio, 
                          learning_rate=0.1, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("--- XGBoost Report ---")
print(classification_report(y_test, y_pred_xgb))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_xgb):.4f}")
print()

xgb_importance_gain = xgb_model.get_booster().get_score(importance_type='gain')

importances_xgb = pd.DataFrame({
    'Feature': xgb_importance_gain.keys(),
    'Gain': xgb_importance_gain.values()
}).sort_values(by='Gain', ascending=False)

print("\n--- XGBoost Feature Gain ---")


In [ ]:

with pd.option_context('display.max_rows', None):
    display(importances_xgb)

In [ ]:
# show all importances (Gain, Cover, Weight)
importance = xgb_model.get_booster().get_score(importance_type='gain')
cover = xgb_model.get_booster().get_score(importance_type='cover')

# create table for comparison 
stats_df = pd.DataFrame([importance, cover], index=['Gain', 'Cover']).T

with pd.option_context('display.max_rows', None):
    display(stats_df)

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_model) 

shap_values = explainer.shap_values(X_test.iloc[:1000])
shap.summary_plot(shap_values, X_test.iloc[:1000])

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(xgb_model, X_test, y_test, n_repeats=10, random_state=42)


In [ ]:
perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance': result.importances_mean
})

perm_df = perm_df.sort_values(by='importance', ascending=False)

print("Top 10 Features (Real Impact):")
print(perm_df.head(10))

print("\nFeatures that detract from the model:")
print(perm_df[perm_df['importance'] < 0])

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

plot_importance(xgb_model, max_num_features=20, importance_type='gain')
plt.show()

In [ ]:
#features_to_drop = ['temp_3pm','temp_9am','dewpoint_3pm','location_Bendigo','location_SalmonGums']
selected_features = importances_xgb[importances_xgb['Gain'] > 15]['Feature'].tolist()
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

In [ ]:


final_model = XGBClassifier(n_estimators=100, scale_pos_weight=ratio, 
                          learning_rate=0.1, max_depth=6, random_state=42)

weights = np.linspace(0.5, 1.0, len(X_train_selected)) # this ensures recent data is prioritized

final_model.fit(X_train_selected, y_train, sample_weight=weights)

y_pred = final_model.predict(X_test_selected)

print("--- final model performance ---")
print(classification_report(y_test, y_pred))
final_f1 = f1_score(y_test, y_pred)
print(f"f1 score: {final_f1:.4f}")
y_prob_final = final_model.predict_proba(X_test_selected)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_final):.4f}")

cm = confusion_matrix(y_test, y_pred)


In [ ]:
#checking overfitting

y_train_pred = final_model.predict(X_train_selected)
f1_train = f1_score(y_train, y_train_pred)

y_test_pred = final_model.predict(X_test_selected)
f1_test = f1_score(y_test, y_test_pred)

print(f"F1 train:    {f1_train:.4f}")
print(f"F1 test:     {f1_test:.4f}")
print(f"difference:  {f1_train - f1_test:.4f}")

In [ ]:
#learning curve
final_model.fit(
    X_train_selected, y_train,
    eval_set=[(X_train_selected, y_train), (X_test_selected, y_test)],
    verbose=False
)
results = final_model.evals_result()
plt.plot(results['validation_0']['logloss'], label='Train')
plt.plot(results['validation_1']['logloss'], label='Test')
plt.title('XGBoost Logloss Curve')
plt.xlabel('trees')
plt.ylabel('logloss')
plt.legend()
plt.show()

### Selecting Models

#### LazyPredict

In [ ]:
import pandas as pd

train = pd.read_csv("../data/preprocessed/train_nan_values.csv")
test = pd.read_csv("../data/preprocessed/test_nan_values.csv")

#sort chronologically if not already done
train = train.sort_values('date').reset_index(drop=True)
X_train = train.drop(columns=['rain_tomorrow','date'])
y_train = train['rain_tomorrow']

X_test = test.drop(columns=['rain_tomorrow','date'])
y_test = test['rain_tomorrow']

from lazypredict import LazyClassifier

clf = LazyClassifier(verbose=0,ignore_warnings=True, custom_metric=None)
models,predictions = clf.fit(X_train, X_test, y_train, y_test)

models

In [ ]:
models.to_csv('../data/lazypredict_models.csv')

#### Best lazypredict models:

In [ ]:
lazypredict=pd.read_csv('../data/lazypredict_models.csv')
lazypredict.sort_values('Balanced Accuracy', ascending=False)

### NearestCentroid

In [ ]:
import pandas as pd
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

train=pd.read_csv('../data/preprocessed/selected_features_train.csv')
test=pd.read_csv('../data/preprocessed/selected_features_test.csv')

X_train=train.drop(['rain_tomorrow'], axis=1)
X_test=test.drop(['rain_tomorrow'], axis=1)
y_train=train['rain_tomorrow']
y_test=test['rain_tomorrow']
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)


# Define scaler groups
cols_std = [
    'min_temp', 'pressure_9am', 'pressure_3pm', 'temp_range',
    'pressure_day_diff', 'dewpoint_spread_9am', 'dewpoint_spread_3pm', 
    'pressure_3pm_24h_diff'
]

cols_norm = [
    'sunshine', 'humidity_3pm', 'cloud_3pm', 
    'lat', 'lon', 'elevation'
]

cols_robust = [
    'rainfall', 'wind_gust_speed', 
    'wind_speed_3pm', 'wind_gust_speed_24h_diff',
    'humidity_3pm_24h_diff'
]

# Setup the Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('std', StandardScaler(), cols_std),
        ('norm', MinMaxScaler(), cols_norm),
        ('robust', RobustScaler(), cols_robust)
    ],
    remainder='passthrough' # For binary columns like rain_today
)

# Create the Pipeline
nc_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', NearestCentroid())
])

# Fit the model
nc_pipeline.fit(X_resampled, y_resampled)

# 5. Predict and Evaluate
y_pred = nc_pipeline.predict(X_test)

print("Nearest Centroid Classification Report:")
print(classification_report(y_test, y_pred))

### XGB feature selecting

In [ ]:
from xgboost import XGBClassifier

train = pd.read_csv("../data/preprocessed/train_nan_values.csv")
test = pd.read_csv("../data/preprocessed/test_nan_values.csv")

#sort chronologically if not already done
train = train.sort_values('date').reset_index(drop=True)
X_train = train.drop(columns=['rain_tomorrow','date'])
y_train = train['rain_tomorrow']

X_test = test.drop(columns=['rain_tomorrow','date'])
y_test = test['rain_tomorrow']

ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_model = XGBClassifier(n_estimators=100, scale_pos_weight=ratio, 
                          learning_rate=0.1, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("--- XGBoost Report ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print()

xgb_importance_gain = xgb_model.get_booster().get_score(importance_type='gain')

importances_xgb = pd.DataFrame({
    'Feature': xgb_importance_gain.keys(),
    'Gain': xgb_importance_gain.values()
}).sort_values(by='Gain', ascending=False)

print("\n--- XGBoost Feature Gain ---")
# show all importances (Gain, Cover)
importance = xgb_model.get_booster().get_score(importance_type='gain')
cover = xgb_model.get_booster().get_score(importance_type='cover')

# create table for comparison 
stats_df = pd.DataFrame([importance, cover], index=['Gain', 'Cover']).T.sort_values('Gain',ascending=False)

with pd.option_context('display.max_rows', None):
    display(stats_df)

In [ ]:
y_train_pred = xgb_model.predict(X_train)
f1_train = f1_score(y_train, y_train_pred)

y_test_pred = xgb_model.predict(X_test)
f1_test = f1_score(y_test, y_test_pred)

print(f"F1 train:    {f1_train:.4f}")
print(f"F1 test:     {f1_test:.4f}")
print(f"difference:  {f1_train - f1_test:.4f}")

### cross validation + hyperparameter search

In [ ]:
## cross validation (time series) and find best params (randomized search)

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
import xgboost as xgb

scale_weight= np.sum(y_train == 0) / np.sum(y_train == 1)

tscv = TimeSeriesSplit(n_splits=5)

model = xgb.XGBClassifier(
    use_label_encoder=False, 
    eval_metric='logloss',
    scale_pos_weight=scale_weight # important for class imbalance
)

param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8],
    'gamma': [0, 0.1, 0.2]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=15,           # tries 15 random combinations
    scoring='f1',        # optimizes on f1 score
    cv=tscv,             # using TimeSeriesSplit
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print(f"best params: {random_search.best_params_}")
print(f"best f1 score: {random_search.best_score_}")

results_df = pd.DataFrame(random_search.cv_results_)

fold_cols = [col for col in results_df.columns if col.startswith('split')]
important_cols = ['params', 'mean_test_score', 'std_test_score'] + fold_cols

best_run_results = results_df[results_df['rank_test_score'] == 1][important_cols]

print("--- performance per fold ---")
print(best_run_results.T)

### find best threshold

In [ ]:
final_model = XGBClassifier(
    **random_search.best_params_, 
    scale_pos_weight=ratio,
    random_state=42,
    tree_method='hist'
)
final_model.fit(X_train, y_train)

y_probs = final_model.predict_proba(X_test)[:, 1]

from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_probs)
f1_scores = 2 * (precision * recall) / (precision + recall)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Final Optimal Threshold: {best_threshold:.3f}")

In [ ]:
final_probs = final_model.predict_proba(X_test)[:, 1]

final_preds = (final_probs >= 0.594).astype(int)

from sklearn.metrics import classification_report, confusion_matrix
print("--- FINALE PERFORMANCE (Threshold 0.594) ---")
print(classification_report(y_test, final_preds))

print(confusion_matrix(y_test, final_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, final_preds):.4f}")

### Resampling

In [ ]:
from sklearn.impute import SimpleImputer

def crossvalidation_resample_threshold_timeseries(X, y, ratio):
    imputer = SimpleImputer(strategy='median')
    
    samplers = {
        "Original (Weighted)": None,
        "Oversampling": RandomOverSampler(sampling_strategy='not majority', random_state=42),
        "SMOTE": SMOTE(random_state=42)
    }

    thresholds = np.arange(0.1, 0.9, 0.01)
    tscv = TimeSeriesSplit(n_splits=5)

    for name, resample in samplers.items():
        seuils, f_scores = [], []
        print(f"--- Strategie: {name} ---")

        for train_index, test_index in tscv.split(X):
            X_train_f, X_val_f = X.iloc[train_index], X.iloc[test_index]
            y_train_f, y_val_f = y.iloc[train_index], y.iloc[test_index]

            if resample is not None:
                X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train_f), columns=X_train_f.columns)
                X_train_res, y_train_res = resample.fit_resample(X_train_imputed, y_train_f)
            else:
                X_train_res, y_train_res = X_train_f, y_train_f

            current_ratio = ratio if resample is None else 1
            model = XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=6, 
                                  scale_pos_weight=current_ratio, random_state=42, tree_method='hist')
            
            model.fit(X_train_res, y_train_res)
            
            y_probs = model.predict_proba(X_val_f)[:, 1]
            
            fold_scores = [f1_score(y_val_f, (y_probs >= t).astype(int)) for t in thresholds]
            ix = np.argmax(fold_scores)
            seuils.append(thresholds[ix])
            f_scores.append(fold_scores[ix])

        print(f"Average Threshold: {np.mean(seuils):.3f}, Average F1: {np.mean(f_scores):.5f}\n")
crossvalidation_resample_threshold_timeseries(X_train, y_train, ratio)

### Calibrating

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

# 1. Get probabilities for the positive class (Rain)
y_probs = final_model.predict_proba(X_test)[:, 1]

# 2. Calculate calibration data
# n_bins=10 splits the probabilities into 10% steps
prob_true, prob_pred = calibration_curve(y_test, y_probs, n_bins=10)

# 3. Plotting
plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=1, label='XGBoost Model')

# The "Perfect Calibration" line (Diagonal)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly Calibrated')

plt.xlabel('Predicted Probability (Model Output)')
plt.ylabel('Actual Rain Frequency (Observed)')
plt.title('Calibration Curve: Model Reliability Check')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, f1_score, roc_auc_score, confusion_matrix

# 1. Apply Calibration (Isotonic is great for large datasets)
# We use cv=5 to avoid using the test set for calibration training
calibrated_xgb = CalibratedClassifierCV(xgb_model, cv=5, method='isotonic')
calibrated_xgb.fit(X_train, y_train)

# 2. Get calibrated probabilities
y_probs_calibrated = calibrated_xgb.predict_proba(X_test)[:, 1]

# 3. Find the new optimal threshold 
thresholds = np.linspace(0, 1, 100)
scores = [f1_score(y_test, (y_probs_calibrated >= t).astype(int)) for t in thresholds]
new_best_threshold = thresholds[np.argmax(scores)]

# 4. Generate final predictions
final_preds = (y_probs_calibrated >= new_best_threshold).astype(int)

# 5. Print Results
print(f"--- Calibrated Model Performance ---")
print(f"New Optimal Threshold: {new_best_threshold:.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_probs_calibrated):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, final_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_preds))

In [ ]:

# 2. Calculate calibration data
# n_bins=10 splits the probabilities into 10% steps
prob_true, prob_pred = calibration_curve(y_test, y_probs_calibrated, n_bins=10)

# 3. Plotting
plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=1, label='Calibrated Model')

# The "Perfect Calibration" line (Diagonal)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly Calibrated')

plt.xlabel('Predicted Probability (Model Output)')
plt.ylabel('Actual Rain Frequency (Observed)')
plt.title('Calibration Curve: Model Reliability Check')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from xgboost import plot_importance
fig, ax = plt.subplots(figsize=(10, 12)) 

plot_importance(final_model, 
                importance_type='gain', 
                ax=ax,
                title='Feature Importance (Gain)')

plt.tight_layout() 
plt.show()

In [ ]:
import matplotlib.pyplot as plt

final_xgb = xgb.XGBClassifier(**random_search.best_params_, scale_pos_weight=ratio, random_state=42, eval_metric="logloss")

eval_set = [(X_train, y_train), (X_test, y_test)]
final_xgb.fit(
    X_train, y_train,
    eval_set=eval_set,
    verbose=False
)

results = final_xgb.evals_result()

plt.figure(figsize=(10, 6))
plt.plot(results['validation_0']['logloss'], label='Train (including scale_pos_weight)')
plt.plot(results['validation_1']['logloss'], label='Test (Generalization)')

plt.axhline(y=min(results['validation_1']['logloss']), color='r', linestyle='--', label='Min Loss Point')

plt.title('Final XGBoost Learning Curve: Logloss over Iterations')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Logloss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(final_model)

shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Feature Impact: What drives Rain in Australia?", fontsize=16)
plt.show()

In [ ]:
import joblib
model_data = {
    'model': calibrated_xgb,
    'threshold': 0.303, 
    'features': list(X_train.columns) 
}

# Speichern
joblib.dump(model_data, "../models/calibrated_xgb_model.pkl")


In [ ]:
#loading the model:
loaded_data = joblib.load("../models/calibrated_weather_model_v1.pkl")
my_model = loaded_data['model']
my_threshold = loaded_data['threshold']

probs = my_model.predict_proba(X)[:, 1]
predictions = (probs >= my_threshold).astype(int)

## LightGBM

In [ ]:
import lightgbm as lgb

model_light = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    class_weight='balanced',
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_light.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

y_pred = model_light.predict(X_test)
y_prob = model_light.predict_proba(X_test)[:, 1]

print("--- LightGBM Report ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print()



In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, f1_score, roc_auc_score, confusion_matrix

# 1. Apply Calibration 
calibrated_light = CalibratedClassifierCV(model_light, cv=5, method='isotonic')
calibrated_light.fit(X_train, y_train)

# 2. Get calibrated probabilities
y_probs_calibrated = calibrated_light.predict_proba(X_test)[:, 1]

# 3. Find the new optimal threshold 
thresholds = np.linspace(0, 1, 100)
scores = [f1_score(y_test, (y_probs_calibrated >= t).astype(int)) for t in thresholds]
new_best_threshold = thresholds[np.argmax(scores)]

# 4. Generate final predictions
final_preds = (y_probs_calibrated >= new_best_threshold).astype(int)

# 5. Print Results
print(f"--- Calibrated Model Performance ---")
print(f"New Optimal Threshold: {new_best_threshold:.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_probs_calibrated):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, final_preds))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, final_preds))

In [ ]:
final_preds = (y_probs_calibrated >= 0.2).astype(int)

from sklearn.metrics import classification_report, confusion_matrix
print("--- FINALE PERFORMANCE (Threshold 0.594) ---")
print(classification_report(y_test, final_preds))

print(confusion_matrix(y_test, final_preds))
print(f"ROC-AUC: {roc_auc_score(y_test, final_preds):.4f}")

## TPOT

I ran TPOT for 13h. This is the result:

In [ ]:
"""import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline, make_union
from tpot.builtins import StackingEstimator
from xgboost import XGBClassifier
from tpot.export_utils import set_param_recursive

train = pd.read_csv('../data/preprocessed/selected_features_train.csv')
test = pd.read_csv('../data/preprocessed/selected_features_test.csv')
training_features = train.drop('rain_tomorrow')
testing_features = test.drop('rain_tomorrow')
training_target = train['rain_tomorrow']
testing_target = test['rain_tomorrow']

# Average CV score on the training set was: 0.6586510035616758
exported_pipeline = make_pipeline(
    StackingEstimator(estimator=ExtraTreesClassifier(bootstrap=True, criterion="gini", max_features=0.9000000000000001, min_samples_leaf=3, min_samples_split=4, n_estimators=100)),
    StackingEstimator(estimator=XGBClassifier(learning_rate=0.1, max_depth=1, min_child_weight=3, n_estimators=100, n_jobs=1, subsample=0.2, verbosity=0)),
    GaussianNB()
)
# Fix random state for all the steps in exported pipeline
set_param_recursive(exported_pipeline.steps, 'random_state', 42)

exported_pipeline.fit(training_features, training_target)
results = exported_pipeline.predict(testing_features)

print("--- TPOT Report ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print()
"""

## SVC

In [ ]:
train_scaled = pd.read_csv('../data/preprocessed/selected_features_train_scaled.csv')
test_scaled = pd.read_csv('../data/preprocessed/selected_features_test_scaled.csv')
X_train_scaled = train_scaled.drop('rain_tomorrow', axis=1)
X_test_scaled = test_scaled.drop('rain_tomorrow', axis=1)
y_train = train_scaled['rain_tomorrow'].reset_index(drop=True)
y_test = test_scaled['rain_tomorrow'].reset_index(drop=True)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, average_precision_score
import seaborn as sns
import numpy as np

# SVC is very slow on large datasets → use a sample
# Full training on 111k rows would take hours
from sklearn.utils import resample

X_train_sample, y_train_sample = resample(
    X_train_scaled, y_train,
    n_samples=20000,
    random_state=42,
    stratify=y_train 
)

# SVC with class_weight due to class imbalance
# probability=True enables threshold optimization via predict_proba
svc = SVC(
    kernel='rbf',            # standard kernel, good for non-linear data
    C=1.0,                   # regularization – higher = less regularization
    gamma='scale',           # automatic: 1 / (n_features * X.var())
    class_weight='balanced',
    probability=True,
    random_state=42
)

svc.fit(X_train_sample, y_train_sample)

# Optimize threshold via precision-recall curve
y_pred_proba = svc.predict_proba(X_test_scaled)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Optimal Threshold: {best_threshold:.3f}")

y_pred_class = (y_pred_proba >= best_threshold).astype(int)

print(classification_report(y_test, y_pred_class))
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_pred_proba):.4f}")
sns.heatmap(confusion_matrix(y_test, y_pred_class), annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Rain', 'Rain'], yticklabels=['No Rain', 'Rain'])

## HCA Clustering

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
import numpy as np

# HCA is O(n²) in memory → use a sample
sample_size = 1000
idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
X_sample = X_train_scaled.iloc[idx]
y_sample = y_train.values[idx]

# PCA for dimensionality reduction (HCA with 148 features is impractical)
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_sample)
print(f"Explained variance with 10 components: {pca.explained_variance_ratio_.sum():.2%}")

# ── Dendrogram (to find optimal number of clusters) ────────────────────
plt.figure(figsize=(12, 5))
linked = linkage(X_pca, method='ward')
dendrogram(linked, truncate_mode='level', p=5,
           color_threshold=0.7 * max(linked[:, 2]))
plt.title('Dendrogram – Hierarchical Clustering')
plt.xlabel('Samples')
plt.ylabel('Ward Distance')
plt.show()

# ── Clustering ─────────────────────────────────────────────────────────
hca = AgglomerativeClustering(
    n_clusters=2,        # 2 because of binary classification (rain / no rain)
    linkage='ward'       # minimizes intra-cluster variance
)
cluster_labels = hca.fit_predict(X_pca)

# ── Visualization (first 2 PCA components) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HCA clusters
axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                c=cluster_labels, cmap='coolwarm', alpha=0.5, s=10)
axes[0].set_title('HCA Clusters')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# True labels for comparison
axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                c=y_sample, cmap='coolwarm', alpha=0.5, s=10)
axes[1].set_title('True Labels (0 = No Rain, 1 = Rain)')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

# ── How well do clusters align with true labels? ──────────────────────
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

print(f"Adjusted Rand Score:    {adjusted_rand_score(y_sample, cluster_labels):.3f}")
print(f"Normalized Mutual Info: {normalized_mutual_info_score(y_sample, cluster_labels):.3f}")

## KMeans Location Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

# ── Step 1: Cluster auf geografischen/klimatischen Features ───────────
# Nur stabile Location-Features nehmen, keine täglichen Wetterwerte
location_features = ['lat', 'lon', 'elevation',
                     'koppen_zone_Equatorial', 'koppen_zone_Grassland',
                     'koppen_zone_Subtropical', 'koppen_zone_Temperate',
                     'koppen_zone_Tropical']

location_features = [f for f in location_features if f in X_train_scaled.columns]

X_train_loc = X_train_scaled[location_features].values
X_test_loc  = X_test_scaled[location_features].values

# Optimale Cluster-Anzahl finden (Elbow Method)
inertias = []
k_range = range(2, 10)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train_loc)
    inertias.append(km.inertia_)

plt.figure()
plt.plot(k_range, inertias, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method – Optimal Number of Location Clusters')
plt.show()

# ── Step 2: Clustering fitten ──────────────────────────────────────────
n_clusters = 4  # nach Elbow anpassen

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
train_clusters = kmeans.fit_predict(X_train_loc)
test_clusters  = kmeans.predict(X_test_loc)

# Cluster-Verteilung prüfen
for c in range(n_clusters):
    n = (train_clusters == c).sum()
    rain_rate = y_train.values[train_clusters == c].mean()
    print(f"Cluster {c}: {n} samples, rain rate: {rain_rate:.2%}")

# ── Step 3: Pro Cluster ein Modell trainieren ─────────────────────────
from sklearn.svm import SVC  # oder XGBoost, LightGBM etc.

cluster_models = {}
cluster_thresholds = {}

for c in range(n_clusters):
    print(f"\nTraining model for Cluster {c}...")
    
    # Train-Daten für diesen Cluster
    mask_train = train_clusters == c
    X_c_train = X_train_scaled[mask_train]
    y_c_train = y_train.values[mask_train]
    
    # Modell trainieren
    model_c = SVC(kernel='rbf', class_weight='balanced',
                  probability=True, random_state=42)
    model_c.fit(X_c_train, y_c_train)
    
    # Threshold optimieren auf Train-Cluster
    from sklearn.metrics import precision_recall_curve
    proba = model_c.predict_proba(X_c_train)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_c_train, proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_threshold = thresholds[np.argmax(f1_scores)]
    
    cluster_models[c] = model_c
    cluster_thresholds[c] = best_threshold
    print(f"  Optimal threshold: {best_threshold:.3f}")

# ── Step 4: Vorhersage auf Test ────────────────────────────────────────
y_pred_all = np.zeros(len(X_test_scaled))

for c in range(n_clusters):
    mask_test = test_clusters == c
    if mask_test.sum() == 0:
        continue
    
    X_c_test = X_test_scaled[mask_test]
    proba = cluster_models[c].predict_proba(X_c_test)[:, 1]
    y_pred_all[mask_test] = (proba >= cluster_thresholds[c]).astype(int)

# ── Step 5: Gesamte Performance ───────────────────────────────────────
print("\n── Overall Performance ───────────────────────────────")
print(classification_report(y_test.values, y_pred_all.astype(int),
                             target_names=['No Rain', 'Rain']))

# Performance pro Cluster
print("\n── Performance per Cluster ───────────────────────────")
for c in range(n_clusters):
    mask_test = test_clusters == c
    if mask_test.sum() == 0:
        continue
    print(f"\nCluster {c} ({mask_test.sum()} samples):")
    print(classification_report(y_test.values[mask_test],
                                 y_pred_all[mask_test].astype(int),
                                 target_names=['No Rain', 'Rain']))

### Appendix B. Integrated Alternate Branch: Dense Neural Network Stress Tests

This appendix integrates the newer dense-network notebook into Notebook 3. It preserves the Sequential, Wide and Deep, and ResNet-style stress tests on the 148-feature NCC-imputed scaled surface. The section is retained as cross-family evidence that static dense networks approached the same performance band without displacing the final CatBoost workflow.


In [ ]:
# For manipulating arrays and DataFrames
import numpy as np
import pandas as pd

# For visualizing performance
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.metrics import classification_report, confusion_matrix, auc
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
# For instantiating a Dense layer and sequential model
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Input, BatchNormalization, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import AUC, Recall, Precision
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2


In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = alpha * tf.pow(1 - p_t, gamma)
        return tf.reduce_mean(focal_weight * bce)
    return loss_fn

In [ ]:
train=pd.read_csv('../data/preprocessed/all_features_ncc_imputed_train_scaled.csv')
test=pd.read_csv('../data/preprocessed/all_features_ncc_imputed_test_scaled.csv')

X_train=train.drop(['rain_tomorrow'], axis=1)
X_test=test.drop(['rain_tomorrow'], axis=1)
y_train=train['rain_tomorrow']
y_test=test['rain_tomorrow']

shape=X_train.shape[1]

### Sequential Model

In [ ]:
# Create the sequential model
model = Sequential()

# Add layers in sequence
model.add(Input(shape=(shape,)))
model.add(Dense(256, activation="relu", kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.4))
model.add(Dense(128, activation="relu", kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.4))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(32, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

early_stop = EarlyStopping(monitor='val_auc_pr', patience=10, mode='max', restore_best_weights=True)

#Display the model's structure
model.summary()

#Compile the model 
model.compile(loss=focal_loss(gamma=2.0, alpha=0.75),
              optimizer=Adam(learning_rate=0.0001),
              metrics=["accuracy",
                        Recall(name="recall"),
                        Precision(name="precision"),
                        AUC(name="auc_roc", curve="ROC"),
                        AUC(name="auc_pr",  curve="PR")
                        ]
                )

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
#class_weight_dict = dict(enumerate(class_weights))
class_weight_dict = {0: 1.0, 1: 6.0}
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                               patience=5, mode='min', min_lr=0.00001)

history=model.fit(X_train,y_train,
                  validation_split=0.15,epochs=500,
                  batch_size=256,
                  callbacks=[early_stop, reduce_lr],
                  class_weight=class_weight_dict)

test_pred=model.predict(X_test)


In [ ]:
#display the evolution of the loss over epochs
plt.figure()
plt.plot(history.history['loss'], label='Loss (training)')
plt.plot(history.history['val_loss'], label='Loss (validation)')
plt.title('Loss per epoch')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
y_pred_proba = model.predict(X_test).flatten()

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Besten F1 finden
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Optimal Threshold: {best_threshold:.3f}")
print(f"Precision: {precisions[best_idx]:.3f}")
print(f"Recall: {recalls[best_idx]:.3f}")
print(f"F1: {f1_scores[best_idx]:.3f}")

In [ ]:
plt.figure()
plt.plot(thresholds, precisions[:-1], label='Precision')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.plot(thresholds, f1_scores[:-1], label='F1')
plt.axvline(x=best_threshold, color='gray', linestyle='--', label=f'Best F1: {best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision / Recall / F1 vs Threshold')
plt.legend()
plt.show()

In [ ]:
threshold=0.42
y_test_class=y_test
y_pred_class=(test_pred > threshold).astype(int).flatten()

#Display a detailed evaluation report of the model's performance
print(classification_report(y_test_class, y_pred_class))
print(f"ROC-AUC: {roc_auc_score(y_test, test_pred):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, test_pred):.4f}")
print("\nConfusion Matrix:")
display(pd.crosstab(y_test_class, y_pred_class,
                  rownames=['Real Class'], colnames=['Predicted Class']))


### TabNet

In [ ]:
"""
from pytorch_tabnet.tab_model import TabNetClassifier

model = TabNetClassifier(
    n_d=32, n_a=32,        # width of attention layers
    n_steps=5,             # number of attention steps
    gamma=1.3,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=1e-3),
    scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
"""

In [ ]:
"""
y_pred_proba = model.predict_proba(X_test)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Optimaler Threshold: {best_threshold:.3f}")

y_pred_class = (y_pred_proba >= best_threshold).astype(int)

print(classification_report(y_test, y_pred_class))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("PR-AUC: ", average_precision_score(y_test, y_pred_proba))
sns.heatmap(confusion_matrix(y_test, y_pred_class), annot=True, fmt='d', cmap='Blues')
"""

### Wide & Deep

In [ ]:
input_layer = Input(shape=(shape,))

# Deep part
deep = Dense(128, activation='relu')(input_layer)
deep = Dense(64, activation='relu')(deep)
deep = Dense(32, activation='relu')(deep)

# Wide part
wide = Dense(32, activation='relu')(input_layer)

# combine
combined = Concatenate()([deep, wide])
output = Dense(1, activation='sigmoid')(combined)

model = Model(inputs=input_layer, outputs=output)

early_stop = EarlyStopping(monitor='val_auc_pr', patience=10, mode='max', restore_best_weights=True)

model.compile(loss="binary_crossentropy",
              optimizer=Adam(learning_rate=0.00001),
              metrics=["accuracy",
                        Recall(name="recall"),
                        Precision(name="precision"),
                        AUC(name="auc_roc", curve="ROC"),
                        AUC(name="auc_pr",  curve="PR")
                        ]
                )

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
#class_weight_dict = dict(enumerate(class_weights))
class_weight_dict = {0: 1.0, 1: 4.0}
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                               patience=5, mode='min', min_lr=0.00001)

history=model.fit(X_train,y_train,
                  validation_split=0.15,epochs=500,
                  batch_size=256,
                  callbacks=[early_stop, reduce_lr],
                  class_weight=class_weight_dict)

test_pred=model.predict(X_test)

In [ ]:
#display the evolution of the loss over epochs
plt.figure()
plt.plot(history.history['loss'], label='Loss (training)')
plt.plot(history.history['val_loss'], label='Loss (validation)')
plt.title('Loss per epoch')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
y_pred_proba = model.predict(X_test).flatten()

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Besten F1 finden
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Optimal Threshold: {best_threshold:.3f}")
print(f"Precision: {precisions[best_idx]:.3f}")
print(f"Recall: {recalls[best_idx]:.3f}")
print(f"F1: {f1_scores[best_idx]:.3f}")

In [ ]:
threshold=0.59
y_test_class=y_test
y_pred_class=(test_pred > threshold).astype(int).flatten()

#Display a detailed evaluation report of the model's performance
print(classification_report(y_test_class, y_pred_class))
print(f"ROC-AUC: {roc_auc_score(y_test, test_pred):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, test_pred):.4f}")
print("Confusion Matrix:")
display(pd.crosstab(y_test_class, y_pred_class,
                  rownames=['Real Class'], colnames=['Predicted Class']))

### ResNet

In [ ]:
from tensorflow.keras.layers import Add, LayerNormalization

input_layer = Input(shape=(shape,))
x = Dense(128, activation='relu')(input_layer)

# Residual Block
residual = x
x = Dense(128, activation='relu')(x)
x = Dense(128, activation='relu')(x)
x = Add()([x, residual])  # Skip Connection
x = LayerNormalization()(x)

x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=input_layer, outputs=output)

early_stop = EarlyStopping(monitor='val_auc_pr', patience=20, mode='max', restore_best_weights=True)

model.compile(loss="binary_crossentropy",
              optimizer=Adam(learning_rate=0.00001),
              metrics=["accuracy",
                        Recall(name="recall"),
                        Precision(name="precision"),
                        AUC(name="auc_roc", curve="ROC"),
                        AUC(name="auc_pr",  curve="PR")
                        ]
                )

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
#class_weight_dict = dict(enumerate(class_weights))
class_weight_dict = {0: 1.0, 1: 4.0}
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                               patience=5, mode='min', min_lr=0.00001)

history=model.fit(X_train,y_train,
                  validation_split=0.15,epochs=500,
                  batch_size=128,
                  callbacks=[early_stop, reduce_lr],
                  class_weight=class_weight_dict)

test_pred=model.predict(X_test)

In [ ]:
#display the evolution of the loss over epochs
plt.figure()
plt.plot(history.history['loss'], label='Loss (training)')
plt.plot(history.history['val_loss'], label='Loss (validation)')
plt.title('Loss per epoch')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
y_pred_proba = model.predict(X_test).flatten()

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Besten F1 finden
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
_sca__f__
print(f"Optimal Threshold: {best_threshold:.3f}")
print(f"Precision: {precisions[best_idx]:.3f}")
print(f"Recall: {recalls[best_idx]:.3f}")
print(f"F1: {f1_scores[best_idx]:.3f}")

In [ ]:
threshold=0.5
y_test_class=y_test
y_pred_class=(test_pred > threshold).astype(int).flatten()

#Display a detailed evaluation report of the model's performance
print(classification_report(y_test_class, y_pred_class))
print(f"ROC-AUC: {roc_auc_score(y_test, test_pred):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, test_pred):.4f}")
print("\nConfusion Matrix:")
display(pd.crosstab(y_test_class, y_pred_class,
                  rownames=['Real Class'], colnames=['Predicted Class']))